In [1]:
# https://github.com/jeandeducla/ML-Time-Series/blob/master/SVM-Accelerometer.ipynb
import pandas as pd
import numpy as np
import datetime as dt

In [2]:
import scipy.stats as st
from scipy.fftpack import fft, fftfreq 
from scipy.signal import argrelextrema
import operator

In [3]:
def stat_area_features(x, Te=1.0):

    mean_ts = np.mean(x, axis=1).reshape(-1,1) # mean
    max_ts = np.amax(x, axis=1).reshape(-1,1) # max
    min_ts = np.amin(x, axis=1).reshape(-1,1) # min
    std_ts = np.std(x, axis=1).reshape(-1,1) # std
    skew_ts = st.skew(x, axis=1).reshape(-1,1) # skew
    kurtosis_ts = st.kurtosis(x, axis=1).reshape(-1,1) # kurtosis 
    iqr_ts = st.iqr(x, axis=1).reshape(-1,1) # interquartile rante
    mad_ts = np.median(np.sort(abs(x - np.median(x, axis=1).reshape(-1,1)),
                               axis=1), axis=1).reshape(-1,1) # median absolute deviation
    area_ts = np.trapz(x, axis=1, dx=Te).reshape(-1,1) # area under curve
    sq_area_ts = np.trapz(x ** 2, axis=1, dx=Te).reshape(-1,1) # area under curve ** 2

    return np.concatenate((mean_ts,max_ts,min_ts,std_ts,skew_ts,kurtosis_ts,
                           iqr_ts,mad_ts,area_ts,sq_area_ts), axis=1)

In [4]:
def frequency_domain_features(x, Te=1.0):

    # As the DFT coefficients and their corresponding frequencies are symetrical arrays
    # with respect to the middle of the array we need to know if the number of readings 
    # in x is even or odd to then split the arrays...
    if x.shape[1]%2 == 0:
        N = int(x.shape[1]/2)
    else:
        N = int(x.shape[1]/2) - 1
    xf = np.repeat(fftfreq(x.shape[1],d=Te)[:N].reshape(1,-1), x.shape[0], axis=0) # frequencies
    dft = np.abs(fft(x, axis=1))[:,:N] # DFT coefficients   
    
    # statistical and area features
    dft_features = stat_area_features(dft, Te=1.0)
    # weighted mean frequency
    dft_weighted_mean_f = np.average(xf, axis=1, weights=dft).reshape(-1,1)
    # 5 first DFT coefficients 
    dft_first_coef = dft[:,:5]    
    # 5 first local maxima of DFT coefficients and their corresponding frequencies
    dft_max_coef = np.zeros((x.shape[0],5))
    dft_max_coef_f = np.zeros((x.shape[0],5))
    for row in range(x.shape[0]):
        # finds all local maximas indexes
        extrema_ind = argrelextrema(dft[row,:], np.greater, axis=0) 
        # makes a list of tuples (DFT_i, f_i) of all the local maxima
        # and keeps the 5 biggest...
        extrema_row = sorted([(dft[row,:][j],xf[row,j]) for j in extrema_ind[0]],
                             key=operator.itemgetter(0), reverse=True)[:5] 
        for i, ext in enumerate(extrema_row):
            dft_max_coef[row,i] = ext[0]
            dft_max_coef_f[row,i] = ext[1]    
    
    return np.concatenate((dft_features,dft_weighted_mean_f,dft_first_coef,
                           dft_max_coef,dft_max_coef_f), axis=1)


In [5]:
def convert_to_list(data, column_name):
    Unique_time = data.timestamp.unique()
    X = []
    for time in Unique_time:
        values = data[data.timestamp==time][column_name].tolist()
        if len(values)<721:
            resample_len=721-len(values)
            new_data = [np.mean(values)] * resample_len
            values.extend(new_data)
        X.append(values)
    return np.array(X)

In [6]:
def make_feature(data, column_name, Te=1.0):
    # convert data into list of lists
    x=convert_to_list(data, column_name)

    # extract statistical feature
    x_stat=stat_area_features(x)
    x_stat=pd.DataFrame(x_stat)
    x_stat.columns=[column_name+'_stat_'+ str(i) for i in range(x_stat.shape[1])]

    # extract statistical feature for the signal difference
    xt_jerk = stat_area_features((x[:, 1:]-x[:, :-1])/Te, Te=Te)
    xt_jerk = pd.DataFrame(xt_jerk)
    xt_jerk.columns=[column_name + '_xt_jerk_' + str(i) for i in range(xt_jerk.shape[1])]
    x_final=pd.concat([x_stat, xt_jerk], axis=1)

    
    # extract frequency_domain_features
    x_domain=frequency_domain_features(x)
    x_domain=pd.DataFrame(x_domain)
    x_domain.columns=[column_name +'_tm_dom_'+ str(i) for i in range(x_domain.shape[1])]
    x_final = pd.concat([x_final, x_domain], axis=1)

    # extract frequency_domain_features for the signal difference
    xf_jerk = frequency_domain_features((x[:, 1:]-x[:, :-1])/Te, Te=1/Te)
    xf_jerk = pd.DataFrame(xf_jerk)
    xf_jerk.columns=[column_name + '_xf_jerk_' + str(i) for i in range(xf_jerk.shape[1])]
    x_final = pd.concat([x_final, xf_jerk], axis=1)

    return x_final

In [8]:
processed_df=pd.read_csv('processed_df.csv')

In [9]:
# test only for humidity column
humidity=make_feature(processed_df, column_name='humidity')

In [10]:
humidity.head()

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_xf_jerk_16,humidity_xf_jerk_17,humidity_xf_jerk_18,humidity_xf_jerk_19,humidity_xf_jerk_20,humidity_xf_jerk_21,humidity_xf_jerk_22,humidity_xf_jerk_23,humidity_xf_jerk_24,humidity_xf_jerk_25
0,50.232676,52.940000,46.270000,1.344530,-0.179787,-0.316444,1.840000,0.880000,36168.594671,1.818200e+06,...,18.803540,16.158123,15.522632,15.068371,12.736747,0.023611,0.055556,0.043056,0.029167,0.040278
1,51.345902,54.720000,46.756667,1.815324,-0.240071,-0.719854,2.866667,1.406667,36968.235519,1.900500e+06,...,12.096034,10.924965,10.469300,10.028296,9.836477,0.076389,0.031944,0.036111,0.138889,0.012500
2,51.500500,56.350000,46.340000,3.162699,0.030752,-1.426342,6.030000,2.996667,37082.796917,1.917108e+06,...,18.250983,16.945170,15.755358,15.423753,15.311718,0.026389,0.019444,0.051389,0.070833,0.022222
3,56.477806,57.170000,55.500000,0.387267,-0.669898,-0.208006,0.463333,0.236667,40664.088903,2.296731e+06,...,4.137086,3.849418,3.084825,3.083419,2.839601,0.040278,0.026389,0.045833,0.008333,0.056944
4,57.071375,57.483333,56.530000,0.207498,-0.277516,-0.693661,0.353333,0.176667,41091.457354,2.345181e+06,...,3.575541,2.576048,2.573024,2.572194,2.501533,0.061111,0.045833,0.054167,0.040278,0.020833


In [11]:
humidity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 578 entries, 0 to 577
Data columns (total 72 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   humidity_stat_0      578 non-null    float64
 1   humidity_stat_1      578 non-null    float64
 2   humidity_stat_2      578 non-null    float64
 3   humidity_stat_3      578 non-null    float64
 4   humidity_stat_4      578 non-null    float64
 5   humidity_stat_5      578 non-null    float64
 6   humidity_stat_6      578 non-null    float64
 7   humidity_stat_7      578 non-null    float64
 8   humidity_stat_8      578 non-null    float64
 9   humidity_stat_9      578 non-null    float64
 10  humidity_xt_jerk_0   578 non-null    float64
 11  humidity_xt_jerk_1   578 non-null    float64
 12  humidity_xt_jerk_2   578 non-null    float64
 13  humidity_xt_jerk_3   578 non-null    float64
 14  humidity_xt_jerk_4   578 non-null    float64
 15  humidity_xt_jerk_5   578 non-null    flo

In [12]:
humidity.describe()

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_xf_jerk_16,humidity_xf_jerk_17,humidity_xf_jerk_18,humidity_xf_jerk_19,humidity_xf_jerk_20,humidity_xf_jerk_21,humidity_xf_jerk_22,humidity_xf_jerk_23,humidity_xf_jerk_24,humidity_xf_jerk_25
count,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,5.780000e+02,...,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000
mean,52.238378,54.661315,49.701401,1.210979,-0.111952,-0.078115,1.771595,0.748484,37611.601531,1.978355e+06,...,10.220886,8.887887,8.107315,7.558769,7.138918,0.034398,0.035683,0.037714,0.039617,0.041080
std,4.082039,3.692574,4.802343,0.878234,0.720605,1.745961,1.518316,0.545647,2939.190260,3.065734e+05,...,5.743357,4.956587,4.475913,4.149439,3.925924,0.018780,0.019424,0.020923,0.024745,0.028666
min,42.658125,44.546667,38.160000,0.129493,-3.685488,-1.880314,0.190000,0.093333,30714.209062,1.311018e+06,...,1.528590,1.367798,1.271257,1.162440,1.087768,0.001389,0.001389,0.001389,0.001389,0.001389
25%,48.926400,51.870000,46.065000,0.531147,-0.509245,-0.889552,0.746667,0.351667,35225.774867,1.725823e+06,...,5.407552,4.700021,4.185563,3.924585,3.722425,0.020833,0.020833,0.022222,0.023611,0.021181
50%,52.137808,54.726667,49.540000,1.013586,-0.126962,-0.492814,1.346667,0.590000,37539.389737,1.957701e+06,...,8.998336,7.923334,7.207592,6.776774,6.538204,0.034722,0.034722,0.036111,0.036111,0.036806
75%,55.440646,57.427500,53.832500,1.604329,0.314928,0.187896,2.244167,0.992500,39917.147823,2.213079e+06,...,13.963449,12.407060,11.114813,10.414392,9.909387,0.047222,0.048611,0.051389,0.051389,0.055556
max,62.125542,69.260000,60.120000,5.396986,2.495298,25.330471,11.290000,3.560000,44732.097771,2.781052e+06,...,31.932356,28.483742,27.348173,23.938733,23.210238,0.136111,0.161111,0.150000,0.150000,0.216667


In [16]:
group_count=pd.read_csv('group_count.csv')

In [25]:
processed_df.drop(columns=processed_df.columns[0], axis=1,  inplace=True)

In [26]:
group_count

,timestamp,countv
0,2019-11-06 12:00:00+01:00,720
1,2019-11-06 16:00:00+01:00,719
2,2019-11-06 20:00:00+01:00,720
3,2019-11-07 00:00:00+01:00,720
4,2019-11-07 04:00:00+01:00,720
...,...,...
573,2020-02-12 16:00:00+01:00,720
574,2020-02-12 20:00:00+01:00,720
575,2020-02-13 00:00:00+01:00,720
576,2020-02-13 04:00:00+01:00,720


In [27]:
# Get for all columns
features_df = group_count.copy()
for column_name in processed_df.columns:
    if column_name not in ("timestamp", 'recordtime'):
        ndf=make_feature(processed_df, column_name=column_name)
        features_df=pd.concat([features_df, ndf], axis=1)
    print(column_name)

temperature
humidity
CO2CosIRValue
CO2MG811Value
MOX1
MOX2
MOX3
MOX4
COValue
timestamp
recordtime


In [28]:
features_df.shape

(578, 650)

In [29]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 578 entries, 0 to 577
Columns: 650 entries, timestamp to COValue_xf_jerk_25
dtypes: float64(648), int64(1), object(1)
memory usage: 2.9+ MB


In [30]:
features_df.describe()

,countv,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,temperature_stat_8,...,COValue_xf_jerk_16,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25
count,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,...,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000
mean,719.012111,20.112026,21.339291,19.029764,0.607713,0.101303,-0.523685,0.967682,0.422640,14480.674897,...,41.632124,34.553082,31.556769,29.638785,28.408412,0.057427,0.070905,0.075197,0.081800,0.082351
std,8.624148,2.175413,2.353628,2.117055,0.434778,0.573439,1.003651,0.828740,0.337929,1566.391339,...,73.408093,63.559711,59.466826,57.534543,56.212440,0.053548,0.056668,0.056345,0.058337,0.054093
min,603.000000,14.248963,15.070000,13.530000,0.070661,-1.956349,-1.814266,0.100000,0.050000,10259.022815,...,13.254871,12.422914,11.847281,11.099621,10.756020,0.001389,0.001389,0.001389,0.001389,0.001389
25%,720.000000,18.778853,19.944167,17.645000,0.300121,-0.219892,-1.093397,0.423333,0.200000,13520.574010,...,19.912966,17.637452,16.383749,15.741140,15.255253,0.005556,0.026389,0.033333,0.034722,0.037500
50%,720.000000,20.113785,20.983333,19.211667,0.514858,0.094387,-0.748464,0.745000,0.323333,14481.543559,...,24.637319,21.563524,20.098567,19.030065,18.288375,0.044444,0.054167,0.058333,0.063194,0.069444
75%,720.000000,21.507425,23.184167,20.337500,0.752071,0.441418,-0.324966,1.182500,0.526667,15485.738966,...,37.665082,31.391257,28.935148,27.909271,26.449353,0.111111,0.120833,0.122222,0.130556,0.127778
max,721.000000,24.929491,26.140000,23.660000,2.527443,2.431825,6.359159,5.333333,2.420000,17949.439745,...,1060.990127,1049.529451,1044.931133,1035.808560,1028.251837,0.229167,0.497222,0.481944,0.490278,0.236111


In [31]:
features_stat_df = pd.DataFrame(features_df.describe())

In [32]:
column__with_variance = features_stat_df.columns[(features_stat_df.loc["std"]/features_stat_df.loc["mean"]) > 0.5]

In [33]:
column__with_variance = list(column__with_variance)
len(column__with_variance), column__with_variance

(436,
 ['temperature_stat_3',
  'temperature_stat_4',
  'temperature_stat_6',
  'temperature_stat_7',
  'temperature_xt_jerk_0',
  'temperature_xt_jerk_1',
  'temperature_xt_jerk_3',
  'temperature_xt_jerk_4',
  'temperature_xt_jerk_5',
  'temperature_xt_jerk_6',
  'temperature_xt_jerk_7',
  'temperature_xt_jerk_8',
  'temperature_xt_jerk_9',
  'temperature_tm_dom_2',
  'temperature_tm_dom_6',
  'temperature_tm_dom_7',
  'temperature_tm_dom_10',
  'temperature_tm_dom_12',
  'temperature_tm_dom_13',
  'temperature_tm_dom_14',
  'temperature_tm_dom_15',
  'temperature_tm_dom_16',
  'temperature_tm_dom_17',
  'temperature_tm_dom_18',
  'temperature_tm_dom_19',
  'temperature_tm_dom_20',
  'temperature_tm_dom_21',
  'temperature_xf_jerk_0',
  'temperature_xf_jerk_1',
  'temperature_xf_jerk_2',
  'temperature_xf_jerk_3',
  'temperature_xf_jerk_5',
  'temperature_xf_jerk_6',
  'temperature_xf_jerk_7',
  'temperature_xf_jerk_8',
  'temperature_xf_jerk_9',
  'temperature_xf_jerk_11',
  'temper

In [34]:
processed_features_df = features_df[column__with_variance].copy()
processed_features_df['timestamp'] = features_df['timestamp']
processed_features_df.dropna(inplace=True)
processed_features_df.timestamp = processed_features_df.timestamp.apply(lambda x: pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S"))
processed_features_df.timestamp = processed_features_df.timestamp.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [74]:
processed_features_df.drop(columns=processed_features_df.columns[0], axis=1,  inplace=True)

In [60]:
processed_features_df.describe()

,temperature_stat_3,temperature_stat_4,temperature_stat_6,temperature_stat_7,temperature_xt_jerk_0,temperature_xt_jerk_1,temperature_xt_jerk_3,temperature_xt_jerk_4,temperature_xt_jerk_5,temperature_xt_jerk_6,...,COValue_xf_jerk_16,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25
count,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,...,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000
mean,0.607713,0.101303,0.967682,0.422640,0.000049,0.502979,0.044568,2.605087,223.860736,0.020611,...,41.632124,34.553082,31.556769,29.638785,28.408412,0.057427,0.070905,0.075197,0.081800,0.082351
std,0.434778,0.573439,0.828740,0.337929,0.001575,0.456304,0.027568,12.612039,207.136350,0.011495,...,73.408093,63.559711,59.466826,57.534543,56.212440,0.053548,0.056668,0.056345,0.058337,0.054093
min,0.070661,-1.956349,0.100000,0.050000,-0.004337,0.026667,0.006187,-26.177831,0.753477,0.003333,...,13.254871,12.422914,11.847281,11.099621,10.756020,0.001389,0.001389,0.001389,0.001389,0.001389
25%,0.300121,-0.219892,0.423333,0.200000,-0.000863,0.176667,0.022583,-1.786506,22.968386,0.010000,...,19.912966,17.637452,16.383749,15.741140,15.255253,0.005556,0.026389,0.033333,0.034722,0.037500
50%,0.514858,0.094387,0.745000,0.323333,-0.000172,0.350778,0.038843,1.853407,173.975718,0.016667,...,24.637319,21.563524,20.098567,19.030065,18.288375,0.044444,0.054167,0.058333,0.063194,0.069444
75%,0.752071,0.441418,1.182500,0.526667,0.000622,0.690676,0.059312,12.843402,395.590849,0.027292,...,37.665082,31.391257,28.935148,27.909271,26.449353,0.111111,0.120833,0.122222,0.130556,0.127778
max,2.527443,2.431825,5.333333,2.420000,0.007958,2.653074,0.171078,25.208983,693.813955,0.056667,...,1060.990127,1049.529451,1044.931133,1035.808560,1028.251837,0.229167,0.497222,0.481944,0.490278,0.236111


In [61]:
df=pd.read_csv('database_pos.csv')

In [62]:
df['timestamp'] = pd.to_datetime(df['datetime'])
df['timestamp'] = df['timestamp'].dt.floor(freq='4H')
df.timestamp = df.timestamp.apply(lambda x: pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S"))
df.timestamp = df.timestamp.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [63]:
move_df = pd.DataFrame(df.groupby('timestamp').sum())
move_df['total_count'] = move_df.sum(axis=1)
move_df.reset_index(inplace=True)

/var/folders/j2/mxbl1d3s7wqdbntq3mdbcgh00000gn/T/ipykernel_30247/847938189.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  move_df = pd.DataFrame(df.groupby('timestamp').sum())


In [67]:
move_df.head()

,timestamp,Living room,Bedroom,Bathroom,Kitchen,Hallway,total_count
0,2019-11-01 00:00:00,0.0,1.0,0.0,0.0,0.0,1.0
1,2019-11-01 04:00:00,3.0,7.0,2.0,0.0,3.0,15.0
2,2019-11-01 08:00:00,11.0,1.0,10.0,5.0,12.0,39.0
3,2019-11-01 12:00:00,2.0,0.0,1.0,0.0,3.0,6.0
4,2019-11-01 16:00:00,7.0,0.0,5.0,2.0,10.0,24.0


In [72]:
processed_features_df=pd.read_csv('processed_features_df.csv')
processed_features_df.timestamp = processed_features_df.timestamp.apply(lambda x: pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S"))
processed_features_df.timestamp = processed_features_df.timestamp.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [75]:
processed_features_df.describe()

,temperature_stat_3,temperature_stat_4,temperature_stat_6,temperature_stat_7,temperature_xt_jerk_0,temperature_xt_jerk_1,temperature_xt_jerk_3,temperature_xt_jerk_4,temperature_xt_jerk_5,temperature_xt_jerk_6,...,COValue_xf_jerk_16,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25
count,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,...,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000,578.000000
mean,0.607713,0.101303,0.967682,0.422640,0.000049,0.502979,0.044568,2.605087,223.860736,0.020611,...,41.632124,34.553082,31.556769,29.638785,28.408412,0.057427,0.070905,0.075197,0.081800,0.082351
std,0.434778,0.573439,0.828740,0.337929,0.001575,0.456304,0.027568,12.612039,207.136350,0.011495,...,73.408093,63.559711,59.466826,57.534543,56.212440,0.053548,0.056668,0.056345,0.058337,0.054093
min,0.070661,-1.956349,0.100000,0.050000,-0.004337,0.026667,0.006187,-26.177831,0.753477,0.003333,...,13.254871,12.422914,11.847281,11.099621,10.756020,0.001389,0.001389,0.001389,0.001389,0.001389
25%,0.300121,-0.219892,0.423333,0.200000,-0.000863,0.176667,0.022583,-1.786506,22.968386,0.010000,...,19.912966,17.637452,16.383749,15.741140,15.255253,0.005556,0.026389,0.033333,0.034722,0.037500
50%,0.514858,0.094387,0.745000,0.323333,-0.000172,0.350778,0.038843,1.853407,173.975718,0.016667,...,24.637319,21.563524,20.098567,19.030065,18.288375,0.044444,0.054167,0.058333,0.063194,0.069444
75%,0.752071,0.441418,1.182500,0.526667,0.000622,0.690676,0.059312,12.843402,395.590849,0.027292,...,37.665082,31.391257,28.935148,27.909271,26.449353,0.111111,0.120833,0.122222,0.130556,0.127778
max,2.527443,2.431825,5.333333,2.420000,0.007958,2.653074,0.171078,25.208983,693.813955,0.056667,...,1060.990127,1049.529451,1044.931133,1035.808560,1028.251837,0.229167,0.497222,0.481944,0.490278,0.236111


In [76]:
processed_features_df['timestamp']

0     2019-11-06 12:00:00
1     2019-11-06 16:00:00
2     2019-11-06 20:00:00
3     2019-11-07 00:00:00
4     2019-11-07 04:00:00
              ...        
573   2020-02-12 16:00:00
574   2020-02-12 20:00:00
575   2020-02-13 00:00:00
576   2020-02-13 04:00:00
577   2020-02-13 08:00:00
Name: timestamp, Length: 578, dtype: datetime64[ns]

In [77]:
processed_features_df = processed_features_df.merge(move_df, how='left', on='timestamp')

In [90]:
features_corr_df = pd.DataFrame(processed_features_df.corr())
features_corr_df.shape

/var/folders/j2/mxbl1d3s7wqdbntq3mdbcgh00000gn/T/ipykernel_30247/3560574771.py:1: FutureWarning: The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  features_corr_df = pd.DataFrame(processed_features_df.corr())


(442, 442)

In [93]:
column__with_corr = features_corr_df.columns[abs(features_corr_df['total_count'])>0.05]
column__with_corr = list(column__with_corr)
len(column__with_corr), column__with_corr

(427,
 ['temperature_stat_3',
  'temperature_stat_4',
  'temperature_stat_6',
  'temperature_stat_7',
  'temperature_xt_jerk_0',
  'temperature_xt_jerk_1',
  'temperature_xt_jerk_3',
  'temperature_xt_jerk_4',
  'temperature_xt_jerk_5',
  'temperature_xt_jerk_6',
  'temperature_xt_jerk_7',
  'temperature_xt_jerk_8',
  'temperature_xt_jerk_9',
  'temperature_tm_dom_2',
  'temperature_tm_dom_7',
  'temperature_tm_dom_12',
  'temperature_tm_dom_16',
  'temperature_tm_dom_17',
  'temperature_tm_dom_18',
  'temperature_tm_dom_19',
  'temperature_tm_dom_20',
  'temperature_tm_dom_21',
  'temperature_xf_jerk_0',
  'temperature_xf_jerk_1',
  'temperature_xf_jerk_3',
  'temperature_xf_jerk_5',
  'temperature_xf_jerk_6',
  'temperature_xf_jerk_7',
  'temperature_xf_jerk_8',
  'temperature_xf_jerk_9',
  'temperature_xf_jerk_11',
  'temperature_xf_jerk_12',
  'temperature_xf_jerk_13',
  'temperature_xf_jerk_14',
  'temperature_xf_jerk_15',
  'temperature_xf_jerk_16',
  'temperature_xf_jerk_17',
  

In [87]:
features_corr_df = pd.DataFrame(processed_features_df[column__with_corr].corr())
features_corr_df

,temperature_stat_3,temperature_stat_4,temperature_stat_6,temperature_stat_7,temperature_xt_jerk_0,temperature_xt_jerk_1,temperature_xt_jerk_3,temperature_xt_jerk_4,temperature_xt_jerk_5,temperature_xt_jerk_6,...,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25,Living room,Bedroom,Bathroom,Kitchen,Hallway,total_count
temperature_stat_3,1.000000,-0.047379,0.958807,0.906890,0.482481,0.230274,0.837719,-0.396026,0.493448,0.399029,...,-0.157948,-0.138343,-0.139079,-0.140984,0.152886,-0.231153,-0.003068,0.306011,0.086199,0.087605
temperature_stat_4,-0.047379,1.000000,-0.052943,-0.073909,-0.359725,-0.027180,0.074279,0.005523,0.059483,-0.049668,...,0.053191,-0.025515,0.027808,-0.000685,-0.044813,-0.026984,-0.095769,0.005225,-0.046036,-0.057269
temperature_stat_6,0.958807,-0.052943,1.000000,0.929264,0.431956,0.178843,0.773775,-0.389625,0.491910,0.327965,...,-0.128026,-0.119643,-0.112189,-0.124488,0.144477,-0.194852,-0.027458,0.269552,0.069946,0.075042
temperature_stat_7,0.906890,-0.073909,0.929264,1.000000,0.392887,0.229491,0.740149,-0.321080,0.468660,0.367865,...,-0.109099,-0.104095,-0.133936,-0.097530,0.109582,-0.150499,-0.029149,0.236921,0.032944,0.052526
temperature_xt_jerk_0,0.482481,-0.359725,0.431956,0.392887,1.000000,-0.411056,0.375783,-0.696613,0.002036,0.248298,...,-0.214373,-0.179923,-0.140745,-0.138088,0.354916,-0.245896,0.232861,0.487586,0.361341,0.326844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Bedroom,-0.231153,-0.026984,-0.194852,-0.150499,-0.245896,0.138771,-0.276759,0.372296,0.051068,-0.198503,...,0.016959,0.107567,0.074877,0.061340,0.241766,1.000000,0.279855,-0.168365,0.191612,0.361240
Bathroom,-0.003068,-0.095769,-0.027458,-0.029149,0.232861,-0.080517,0.067552,-0.147975,-0.163914,0.153795,...,-0.166373,-0.135587,-0.101985,-0.112323,0.699642,0.279855,1.000000,0.411706,0.743847,0.831079
Kitchen,0.306011,0.005225,0.269552,0.236921,0.487586,-0.328445,0.322201,-0.532143,-0.136194,0.249507,...,-0.267832,-0.253406,-0.222305,-0.225353,0.649367,-0.168365,0.411706,1.000000,0.633885,0.646789
Hallway,0.086199,-0.046036,0.069946,0.032944,0.361341,-0.236938,0.130176,-0.336285,-0.183817,0.142287,...,-0.247590,-0.178054,-0.151737,-0.189083,0.898494,0.191612,0.743847,0.633885,1.000000,0.945696


In [85]:
# remove highly correlated variables
drop_list = []
keep_list = []
for clmn in column__with_corr:
    if not clmn in drop_list:
        clmns = features_corr_df.loc[clmn]
        clmns = list(clmns[clmns>0.8].index)

        if len(clmns) > 1:
            clmns = [x for x in clmns if x != clmn]
            drop_list.extend(clmns)
            
        keep_list.append(clmn)
    

In [86]:
len(drop_list), len(keep_list)

(497, 168)